# B5：Developer Copilot — 代碼助理架構設計

> **場景：** 企業內部 Developer Copilot，讓工程師用自然語言查詢內部程式碼庫、生成代碼、跑測試、解釋錯誤。  
> **核心挑戰：** 代碼庫太大無法全部放入 Context；生成的代碼需要安全執行；工程師要即時看到輸出。

## 核心架構決策

| 決策 | 選擇 | 拒絕的方案 |
|------|------|------------|
| 代碼庫索引 | AST-aware Chunking + Symbol Index | 整個 repo 餵給 LLM |
| 代碼執行 | 沙盒隔離（Docker/E2B） | 直接在 host 執行 |
| 回應方式 | Streaming | 等全部生成完再顯示 |
| Context 策略 | 動態 Context 選擇（相關文件 + 引用） | 整個文件塞進去 |
| 錯誤修正 | Agentic Loop（自動測試 + 修正） | 只生成代碼不驗證 |

In [ ]:
import ast
import re
import asyncio
import subprocess
import tempfile
import os
import time
from typing import Optional
from dataclasses import dataclass, field

try:
    from dotenv import load_dotenv; load_dotenv()
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import HumanMessage, SystemMessage
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, streaming=True)
    LLM_AVAILABLE = True
    print("✅ LLM 可用（含 Streaming）")
except:
    LLM_AVAILABLE = False
    print("⚠️  LLM 未設定")

# 模擬企業代碼庫
SAMPLE_CODEBASE = {
    "src/auth/jwt_handler.py": """
import jwt
from datetime import datetime, timedelta

SECRET_KEY = "your-secret"

def generate_token(user_id: str, expires_hours: int = 24) -> str:
    payload = {
        'user_id': user_id,
        'exp': datetime.utcnow() + timedelta(hours=expires_hours)
    }
    return jwt.encode(payload, SECRET_KEY, algorithm='HS256')

def verify_token(token: str) -> dict:
    return jwt.decode(token, SECRET_KEY, algorithms=['HS256'])
""",
    "src/database/user_repo.py": """
from sqlalchemy.orm import Session
from models import User

def get_user_by_id(db: Session, user_id: str) -> User:
    return db.query(User).filter(User.id == user_id).first()

def create_user(db: Session, email: str, hashed_password: str) -> User:
    user = User(email=email, hashed_password=hashed_password)
    db.add(user)
    db.commit()
    return user
""",
    "src/api/user_routes.py": """
from fastapi import APIRouter, Depends
from auth.jwt_handler import verify_token
from database.user_repo import get_user_by_id

router = APIRouter()

@router.get("/users/{user_id}")
async def get_user(user_id: str, token: str = Depends(verify_token)):
    user = get_user_by_id(db, user_id)
    return user
"""
}

print(f"模擬代碼庫：{len(SAMPLE_CODEBASE)} 個文件")

## 決策 1：AST-aware Chunking — 為什麼不整個文件塞進去？

**❌ 被拒絕：把整個 repo 或整個文件給 LLM**  
一個大型 repo 有 1,000+ 個文件，10MB+ 的代碼。  
即使用 Gemini 的 1M context，每次查詢都塞 10MB = 巨大成本。  
而且大部分代碼和用戶的問題不相關。

**✅ 選擇：AST 解析 + Symbol Index + 動態 Context 選擇**

In [ ]:
@dataclass
class CodeSymbol:
    """代碼中的一個符號（函數、類別）"""
    name: str
    symbol_type: str    # function | class | method
    file_path: str
    line_start: int
    line_end: int
    source_code: str
    docstring: str = ""
    imports: list[str] = field(default_factory=list)
    calls: list[str] = field(default_factory=list)  # 這個函數呼叫了哪些其他函數


class ASTChunker:
    """
    AST-aware Chunking：按程式語言的語意單位切割
    
    為什麼按 AST 切割，不按行數切割？
    按行數切割的問題：
    → 可能在函數中間切割，破壞代碼結構
    → LLM 看到的是殘缺的函數，難以理解
    → 函數間的引用關係丟失
    
    AST 切割的優點：
    → 每個 chunk 是完整的語意單位（函數、類別）
    → 自動提取函數簽名、文件字符串、呼叫關係
    → Symbol Index 讓「找到 verify_token 的實作」瞬間完成
    """
    
    def parse_python_file(self, file_path: str, source_code: str) -> list[CodeSymbol]:
        """用 AST 解析 Python 文件，提取所有函數和類別"""
        symbols = []
        lines = source_code.split('\n')
        
        try:
            tree = ast.parse(source_code)
        except SyntaxError:
            return []
        
        # 提取所有 import
        imports = [
            ast.unparse(node) for node in ast.walk(tree)
            if isinstance(node, (ast.Import, ast.ImportFrom))
        ]
        
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                symbol_type = "class" if isinstance(node, ast.ClassDef) else "function"
                
                # 提取函數體
                start = node.lineno - 1
                end = node.end_lineno
                source = '\n'.join(lines[start:end])
                
                # 提取文件字符串
                docstring = ""
                if (isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))
                        and node.body
                        and isinstance(node.body[0], ast.Expr)
                        and isinstance(node.body[0].value, ast.Constant)):
                    docstring = node.body[0].value.value
                
                # 提取函數呼叫（找出依賴關係）
                calls = [
                    call.func.id if isinstance(call.func, ast.Name) else
                    (call.func.attr if isinstance(call.func, ast.Attribute) else "")
                    for call in ast.walk(node)
                    if isinstance(call, ast.Call) and hasattr(call, 'func')
                ]
                calls = list(set(filter(None, calls)))
                
                symbols.append(CodeSymbol(
                    name=node.name,
                    symbol_type=symbol_type,
                    file_path=file_path,
                    line_start=node.lineno,
                    line_end=node.end_lineno,
                    source_code=source,
                    docstring=docstring,
                    imports=imports,
                    calls=calls
                ))
        return symbols


class SymbolIndex:
    """Symbol 快速查找索引"""
    
    def __init__(self):
        self.symbols: dict[str, CodeSymbol] = {}    # name → Symbol
        self.file_symbols: dict[str, list[str]] = {}  # file → [symbol names]
        self.call_graph: dict[str, list[str]] = {}   # symbol → [called by]
    
    def index_symbols(self, symbols: list[CodeSymbol]):
        for sym in symbols:
            self.symbols[sym.name] = sym
            self.file_symbols.setdefault(sym.file_path, []).append(sym.name)
            for called in sym.calls:
                self.call_graph.setdefault(called, []).append(sym.name)
    
    def find_symbol(self, name: str) -> Optional[CodeSymbol]:
        return self.symbols.get(name)
    
    def find_usages(self, symbol_name: str) -> list[str]:
        """找出哪些函數呼叫了這個 symbol"""
        return self.call_graph.get(symbol_name, [])


# 建立索引
chunker = ASTChunker()
symbol_index = SymbolIndex()

all_symbols = []
for file_path, source in SAMPLE_CODEBASE.items():
    symbols = chunker.parse_python_file(file_path, source)
    all_symbols.extend(symbols)

symbol_index.index_symbols(all_symbols)

print("AST Symbol Index 建立完成：")
print(f"  索引了 {len(all_symbols)} 個 Symbols")
for sym in all_symbols:
    calls_str = f" | 呼叫: {sym.calls}" if sym.calls else ""
    print(f"  [{sym.symbol_type}] {sym.name} ({sym.file_path}:{sym.line_start}){calls_str}")

print("\n查詢：verify_token 在哪裡被使用？")
usages = symbol_index.find_usages("verify_token")
print(f"  被 {usages} 呼叫")
print("\n查詢：verify_token 的完整實作？")
sym = symbol_index.find_symbol("verify_token")
if sym:
    print(f"  在 {sym.file_path}:{sym.line_start}")
    print(f"  {sym.source_code}")

## 決策 2：沙盒代碼執行 — 為什麼不直接執行？

In [ ]:
class CodeSandbox:
    """
    沙盒代碼執行環境
    
    為什麼不直接在 host 執行 LLM 生成的代碼？
    
    風險：
    1. LLM 可能生成惡意代碼（刪除文件、讀取私密資料）
    2. 用戶可能輸入「幫我寫一段刪除 /var 的代碼」
    3. 代碼 bug 可能影響 host 系統
    
    沙盒方案比較：
    
    1. subprocess + 資源限制（本 notebook 使用）
       優點：簡單，不需要額外服務
       缺點：不完全隔離（共用 filesystem namespace）
       適合：低風險場景（只跑 Python 腳本）
    
    2. Docker 容器
       優點：完全隔離，可以限制網路和 filesystem
       缺點：啟動延遲（1-3s），需要 Docker daemon
       適合：生產環境，中等風險場景
    
    3. E2B（sandboxed execution cloud）
       優點：專門為 AI code execution 設計，API 呼叫
       缺點：外部服務依賴，有網路延遲
       適合：需要快速開發，不想自管 Docker
    
    4. Firecracker / gVisor（Google Cloud 使用）
       優點：microVM 隔離，延遲接近原生
       缺點：複雜度高
       適合：高風險場景（允許使用者執行任意代碼）
    """
    
    MAX_EXEC_TIME = 10    # 秒
    MAX_OUTPUT_LEN = 5000 # 字符
    
    # 危險操作黑名單（額外的靜態分析層）
    DANGEROUS_PATTERNS = [
        r'import\s+os\b.*system',
        r'subprocess\.(?:call|run|Popen)',
        r'open\([\'"].*[\'"\]]\s*[,)][^)]*[\'"]w[\'"]',  # open for write
        r'shutil\.rmtree',
        r'__import__',
        r'eval\s*\(',
        r'exec\s*\(',
    ]
    
    def static_check(self, code: str) -> dict:
        """執行前的靜態安全檢查"""
        for pattern in self.DANGEROUS_PATTERNS:
            if re.search(pattern, code, re.IGNORECASE):
                return {"safe": False, "reason": f"包含危險模式：{pattern[:40]}"}
        try:
            ast.parse(code)  # 語法檢查
        except SyntaxError as e:
            return {"safe": False, "reason": f"語法錯誤：{e}"}
        return {"safe": True, "reason": ""}
    
    def execute(self, code: str, test_input: str = "") -> dict:
        """在隔離環境中執行代碼"""
        # 靜態安全檢查
        check = self.static_check(code)
        if not check["safe"]:
            return {"success": False, "error": check["reason"], "output": ""}
        
        # 用 tempfile 建立隔離環境
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py',
                                          delete=False, dir='/tmp') as f:
            f.write(code)
            temp_path = f.name
        
        try:
            result = subprocess.run(
                ['python3', temp_path],
                capture_output=True,
                text=True,
                timeout=self.MAX_EXEC_TIME,
                # 生產中加入：
                # - cgroup 限制 CPU/Memory
                # - seccomp filter 限制 syscall
                # - 或用 Docker/E2B 替換
            )
            output = (result.stdout + result.stderr)[:self.MAX_OUTPUT_LEN]
            return {
                "success": result.returncode == 0,
                "output": output,
                "error": result.stderr[:500] if result.returncode != 0 else ""
            }
        except subprocess.TimeoutExpired:
            return {"success": False, "error": f"執行超時（>{self.MAX_EXEC_TIME}s）", "output": ""}
        finally:
            os.unlink(temp_path)


sandbox = CodeSandbox()

test_codes = [
    ("正常代碼",
     "def fibonacci(n):\n    if n <= 1: return n\n    return fibonacci(n-1) + fibonacci(n-2)\nprint(fibonacci(10))"),
    ("危險：exec",
     "exec('print(\"hacked\")')"),
    ("危險：subprocess",
     "import subprocess\nsubprocess.run(['ls', '-la'])"),
    ("正常：數學計算",
     "result = sum(i**2 for i in range(100))\nprint(f'結果：{result}')"),
]

print("沙盒執行測試：")
print("=" * 55)
for label, code in test_codes:
    result = sandbox.execute(code)
    icon = "✅" if result["success"] else "🚫"
    print(f"\n[{label}]")
    print(f"  {icon} success={result['success']}")
    if result["output"]:
        print(f"  output: {result['output'].strip()[:50]}")
    if result["error"]:
        print(f"  error: {result['error'][:60]}")

## 決策 3：Agentic 代碼修正迴圈

In [ ]:
async def code_generation_agent(user_request: str, context_symbols: list[CodeSymbol],
                                  max_iterations: int = 3) -> dict:
    """
    Agentic 代碼生成迴圈：
    1. 生成代碼
    2. 在沙盒中執行
    3. 如果失敗，把錯誤回饋給 LLM 修正
    4. 最多迭代 max_iterations 次
    
    為什麼不只生成一次代碼？
    → LLM 生成的代碼第一次成功率約 70-80%
    → 自動修正後，累計成功率可達 90%+
    → 對工程師來說：不用看到失敗，直接拿到能跑的代碼
    """
    
    print(f"\nAgentic 代碼生成：'{user_request}'")
    
    # 建立 Context（相關的 Symbol 代碼）
    context_str = "\n".join(
        f"# {sym.file_path}\n{sym.source_code}"
        for sym in context_symbols
    )
    
    code = ""
    for iteration in range(max_iterations):
        print(f"\n[Iteration {iteration+1}] 生成代碼...")
        
        if LLM_AVAILABLE:
            if iteration == 0:
                prompt = f"""參考以下現有代碼：
{context_str}

請生成 Python 代碼完成以下任務（只輸出代碼，可以加 print 顯示結果）：
{user_request}"""
            else:
                prompt = f"""以下代碼執行失敗：
{code}

錯誤：{error_msg}

請修正代碼（只輸出修正後的代碼）："""
            
            resp = await llm.ainvoke([HumanMessage(content=prompt)])
            code = resp.content.strip().strip("```python").strip("```").strip()
        else:
            # 模擬：第一次故意寫錯，第二次修正
            if iteration == 0:
                code = "print(undefined_variable)  # 故意的錯誤"
            else:
                code = "result = 1 + 1\nprint(f'1 + 1 = {result}')"
        
        print(f"  生成代碼：{code[:60]}")
        
        # 在沙盒中執行
        exec_result = sandbox.execute(code)
        
        if exec_result["success"]:
            print(f"  ✅ 執行成功！輸出：{exec_result['output'].strip()[:50]}")
            return {"success": True, "code": code, "output": exec_result["output"],
                    "iterations": iteration + 1}
        else:
            error_msg = exec_result["error"]
            print(f"  ❌ 執行失敗：{error_msg[:60]}")
            if iteration < max_iterations - 1:
                print(f"  🔄 自動修正中...")
    
    return {"success": False, "code": code, "error": error_msg, "iterations": max_iterations}


# 測試：從 Symbol Index 找相關代碼，注入 Context
relevant_symbols = [
    symbol_index.find_symbol("generate_token"),
    symbol_index.find_symbol("verify_token")
]
relevant_symbols = [s for s in relevant_symbols if s]

result = await code_generation_agent(
    user_request="生成一個 JWT token，然後立刻驗證它，印出解碼後的 payload",
    context_symbols=relevant_symbols,
    max_iterations=3
)

print(f"\n最終結果：{'成功' if result['success'] else '失敗'}")
print(f"迭代次數：{result['iterations']}")

In [ ]:
print("""
FDE 面試：Developer Copilot 架構決策
═══════════════════════════════════════════════════════

Q: 為什麼要 AST-aware Chunking，不直接把文件整個給 LLM？
A: 大型代碼庫有 1,000+ 個文件，不可能全塞進 Context。
   即使有 1M context 的 Gemini，每次查詢都付 10MB 的成本不現實。
   AST 解析的優點：
   1. 每個 chunk 是完整的語意單位（函數/類別），不會被截斷
   2. Symbol Index 讓「找到 verify_token」從 O(n) 降到 O(1)
   3. Call Graph 讓「哪些地方用到了這個函數」瞬間回答
   4. 只注入和查詢相關的 Symbols，Context 精準、成本低

Q: 代碼執行為什麼需要沙盒？不能靠 Prompt 說「不要生成危險代碼」？
A: 代碼執行的安全不能靠 Prompt。
   原因：
   1. LLM 可能生成有 bug 的代碼（不是惡意，但會影響系統）
   2. Prompt Injection：用戶可能輸入要求 LLM 生成危險代碼的請求
   3. 即使 LLM 拒絕，有人可能繞過
   沙盒層次：
   - 靜態分析（執行前）：AST 掃描危險模式
   - 執行隔離：timeout + subprocess 或 Docker/E2B
   - 生產建議：E2B 或 Docker（共用 filesystem 仍有風險）

Q: Agentic 修正迴圈最多迭代幾次？為什麼？
A: 2-3 次。
   第 1 次失敗通常是小語法錯誤或 API 理解問題，修正後大多成功。
   第 2 次失敗通常是邏輯問題，LLM 可能理解了但設計有問題。
   超過 3 次：問題可能是 Context 不夠（需要更多相關代碼），
   或者是 Task 本身超出範圍，應該回傳「需要人工協助」。
   每次迭代有成本，不能無限重試。
═══════════════════════════════════════════════════════
""")